# Pathrise Placement Classification

This educational project explores whether information recorded at the beginning of a job-search program can predict whether a participant is eventually placed. The exact collection time of each field is not documented, so this is an assumption to verify before any real-world use. This model is not intended for decisions about individuals.

We keep the original analysis separately and create a reproducible workflow with preprocessing inside the model pipeline.

## 1. Load and inspect the data

The Excel workbook must be in the same folder as this notebook.

In [1]:
import pandas as pd
import numpy as np

raw = pd.read_excel('Data_Pathrise.xlsx')
print('Shape:', raw.shape)
display(raw.head())
display(raw.isna().sum().sort_values(ascending=False))

Shape: (2544, 16)


,id,pathrise_status,primary_track,cohort_tag,program_duration_days,placed,employment_status,highest_level_of_education,length_of_job_search,biggest_challenge_in_search,professional_experience,work_authorization_status,number_of_interviews,number_of_applications,gender,race
0,1,Active,SWE,OCT19A,NaN,0,Unemployed,Bachelor's Degree,3-5 months,Hearing back on my applications,3-4 years,Canada Citizen,2.0,900,Male,Non-Hispanic White or Euro-American
1,2,Active,PSO,JAN20A,NaN,0,Unemployed,"Some College, No Degree",3-5 months,Getting past final round interviews,1-2 years,Citizen,6.0,0,Male,Non-Hispanic White or Euro-American
2,3,Closed Lost,Design,AUG19B,0.0,0,Employed Part-Time,Master's Degree,Less than one month,Figuring out which jobs to apply for,Less than one year,Citizen,0.0,0,Male,East Asian or Asian American
3,4,Closed Lost,PSO,AUG19B,0.0,0,Contractor,Bachelor's Degree,Less than one month,Getting past final round interviews,Less than one year,Citizen,5.0,25,Male,Decline to Self Identify
4,5,Placed,SWE,AUG19A,89.0,1,Unemployed,Bachelor's Degree,1-2 months,Hearing back on my applications,1-2 years,F1 Visa/OPT,10.0,100,Male,East Asian or Asian American


program_duration_days          616
gender                         492
work_authorization_status      284
employment_status              229
professional_experience        222
number_of_interviews           218
length_of_job_search            74
highest_level_of_education      58
biggest_challenge_in_search     24
race                            18
cohort_tag                       8
id                               0
pathrise_status                  0
primary_track                    0
placed                           0
number_of_applications           0
dtype: int64

## 2. Define the study population and features

Exclude participants with ongoing or unresolved statuses (`Active`, `Deferred`, `MIA`, `Break`), as their final outcome is not known. Withdrawals are retained as non-placement, following the dataset label. This is a study design choice, not a causal conclusion, and may bias results.

Exclude identifiers, the outcome-related status, program duration, interview and application counts, and cohort tag. These may reveal later events or have unclear timing. Gender and race are omitted as inputs; this does not guarantee fairness. The remaining fields are assumed to be available at the beginning, which must be confirmed against the data documentation.

In [2]:
unresolved = ['Active', 'Deferred', 'MIA', 'Break']
data = raw.loc[~raw['pathrise_status'].isin(unresolved)].copy()
data.columns = data.columns.str.strip()
features = [
    'primary_track', 'employment_status', 'highest_level_of_education',
    'length_of_job_search', 'biggest_challenge_in_search',
    'professional_experience', 'work_authorization_status'
]
X = data[features].copy()
y = data['placed'].astype(int)
print('Rows:', len(data))
print('Class counts:')
print(y.value_counts().sort_index())
print('Missing values:')
print(X.isna().sum())
assert set(y.unique()) == {0, 1}
assert X.index.equals(y.index)

Rows: 1894
Class counts:
placed
0    938
1    956
Name: count, dtype: int64
Missing values:
primary_track                    0
employment_status              177
highest_level_of_education      40
length_of_job_search            65
biggest_challenge_in_search     20
professional_experience        156
work_authorization_status      220
dtype: int64


## 3. Split the data

Reserve 20% of rows for a final test; keep class proportions similar. The test set is not used to choose the model.

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Training rows:', len(X_train), '| Test rows:', len(X_test))

Training rows: 1515 | Test rows: 379


## 4. Build pipelines and compare models

Missing categories are filled with a constant and one-hot encoding is learned inside each cross-validation fold. Unknown categories at prediction time are safely ignored. We compare a majority-class baseline, logistic regression, and random forest using stratified five-fold cross-validation. Accuracy and F1 are shown; model selection uses F1 for the placed class.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

def build_pipeline(classifier):
    preprocessing = ColumnTransformer([
        ('categories', Pipeline([
            ('missing', SimpleImputer(strategy='constant', fill_value='Unknown')),
            ('encode', OneHotEncoder(handle_unknown='ignore'))
        ]), features)
    ])
    return Pipeline([('preprocess', preprocessing), ('classifier', classifier)])

models = {
    'Majority baseline': build_pipeline(DummyClassifier(strategy='most_frequent')),
    'Logistic regression': build_pipeline(LogisticRegression(max_iter=1000, random_state=42)),
    'Random forest': build_pipeline(RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=42, n_jobs=1))
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring={'accuracy':'accuracy', 'f1':'f1'}, error_score='raise', n_jobs=1)
    rows.append({'Model': name, 'CV accuracy': scores['test_accuracy'].mean(),
                 'CV accuracy std': scores['test_accuracy'].std(), 'CV F1': scores['test_f1'].mean()})
comparison = pd.DataFrame(rows).sort_values('CV F1', ascending=False).reset_index(drop=True)
display(comparison.round(3))

,Model,CV accuracy,CV accuracy std,CV F1
0,Majority baseline,0.505,0.000,0.671
1,Logistic regression,0.545,0.018,0.582
2,Random forest,0.535,0.016,0.559


## 5. Fit the selected model and evaluate once on the test set

F1 balances precision and recall for the placed class. The majority baseline is included in the comparison; selection below is restricted to the two predictive models. A single split is not evidence of performance on future cohorts.

In [5]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

selected_name = comparison.loc[comparison['Model'] != 'Majority baseline', 'Model'].iloc[0]
selected_model = models[selected_name]
selected_model.fit(X_train, y_train)
predictions = selected_model.predict(X_test)
probabilities = selected_model.predict_proba(X_test)[:, 1]
print('Selected model:', selected_name)
print(f'Test accuracy: {accuracy_score(y_test, predictions):.3f}')
print(f'Test ROC AUC: {roc_auc_score(y_test, probabilities):.3f}')
print('Confusion matrix:')
print(confusion_matrix(y_test, predictions, labels=[0, 1]))
print(classification_report(y_test, predictions, target_names=['Not placed', 'Placed'], zero_division=0))

Selected model: Logistic regression
Test accuracy: 0.559
Test ROC AUC: 0.565
Confusion matrix:
[[ 90  98]
 [ 69 122]]
              precision    recall  f1-score   support

  Not placed       0.57      0.48      0.52       188
      Placed       0.55      0.64      0.59       191

    accuracy                           0.56       379
   macro avg       0.56      0.56      0.56       379
weighted avg       0.56      0.56      0.56       379



## 6. Save and verify the complete pipeline

The saved artifact includes preprocessing and expects a DataFrame with the original feature names. Only load joblib files from trusted sources.

In [6]:
from pathlib import Path
import joblib
import sklearn

Path('models').mkdir(exist_ok=True)
model_path = Path('models/pathrise_pipeline.joblib')
artifact = {'pipeline': selected_model, 'feature_columns': features,
            'model_name': selected_name, 'sklearn_version': sklearn.__version__}
joblib.dump(artifact, model_path)
reloaded = joblib.load(model_path)
assert np.array_equal(predictions, reloaded['pipeline'].predict(X_test[features]))
print('Saved and verified:', model_path)

Saved and verified: models\pathrise_pipeline.joblib


## Limitations and next steps

- Confirm the dataset source, license, label definitions and when every feature is recorded before publication or real-world use.
- Excluding unresolved cases and treating withdrawals as not placed can introduce selection bias.
- Random row-based splitting does not measure performance on future cohorts. A time-based holdout and, if applicable, participant-grouped evaluation are important next steps.
- The test data were used for this report; future model changes require a fresh evaluation strategy.
- Omitting protected attributes does not guarantee fairness; evaluate disparities responsibly if appropriate data and permissions are available.
- This is educational analysis and is not suitable for placement or other consequential decisions.